# Unified protocol results — Run All

**Runtime → Run all.** No prompts, no cells to skip, no GPU needed.

**What this does.** Re-scores every method — Direct NER (Exp01), three-step cascade (Exp04), cascade + repair (Exp05), confidence fusion (Exp06), and all five OOF routers — on the *same* nested-CV outer-test tokens, then writes `unified_protocol_table.xlsx` to the run directory on Drive. That single workbook holds the main table, the fusion-vs-baseline deltas, the Holm-adjusted p-values, and the token-level error breakdown.

**What this does NOT do.** It never trains anything. Every OOF run already retrained Exp01/Exp04 per outer fold and stored both predictions next to gold in its `detailed_results` sheet, so this is pure post-processing. That is why Run All is safe here and is *not* safe in `colab_150_sentences_journal.ipynb`, which contains the training cell.

**Prerequisite.** A completed training run must already exist on Drive under `OUTPUT_DIR` (cell 2). Section 4 checks this and stops with a clear message rather than running for 20 minutes and producing nothing.

**Expect roughly 20 minutes**, almost all of it reading ~900 workbooks over the Google Drive mount. Progress prints as it goes and is mirrored to `cell9b_unified.log`.

## 1 — Mount Google Drive

In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running on Colab — skipping the Drive mount.")

## 2 — Paths

`OUTPUT_DIR` must point at the directory the training run wrote to. Everything else is derived.

In [ ]:
import os
from pathlib import Path

WORK_ROOT = Path("/content/drive/MyDrive/thesis_project")
OUTPUT_DIR = WORK_ROOT / "5folds_run"

# Code on the Colab disk, not Drive: git checkouts on a Drive mount often break.
REPO_DIR = Path("/content/thesis")
REPO_URL = "https://github.com/menachemsperka/thesis.git"

NER_DATASET_PKL = REPO_DIR / "data" / "ner_dataset_150_seed42.pkl"
SPLITS_DIR = OUTPUT_DIR / "exp07" / "splits"
RUNNER_XLSX = OUTPUT_DIR / "cross_comparison_latest.xlsx"
UNIFIED_XLSX = OUTPUT_DIR / "unified_protocol_table.xlsx"

print("OUTPUT_DIR:  ", OUTPUT_DIR)
print("REPO_DIR:    ", REPO_DIR)
print("UNIFIED_XLSX:", UNIFIED_XLSX)

## 3 — Clone or update the repository, then install dependencies

The repo is public, so no token. The 150-sentence pickle ships under `data/`, so nothing to upload.

In [ ]:
import shutil
import subprocess


def _git_head_ok(repo):
    if not (repo / ".git").is_dir():
        return False
    return (
        subprocess.run(
            ["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True
        ).returncode
        == 0
    )


# Step out of REPO_DIR first: on a re-run the cwd is inside it, and rmtree would
# leave the process on a deleted directory so the clone that follows would fail.
os.chdir(REPO_DIR.parent)

if _git_head_ok(REPO_DIR):
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", "main"], check=True)
    print("Updated existing clone.")
else:
    if REPO_DIR.is_dir():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", "main", "--single-branch",
            REPO_URL, str(REPO_DIR),
        ],
        check=True,
    )
    print("Cloned repository.")

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)

for required in (REPO_DIR / "build_unified_protocol_table.py", NER_DATASET_PKL):
    if not required.is_file():
        raise FileNotFoundError(
            f"Missing {required} in the clone. Push it to origin/main, then re-run this cell."
        )
print("Builder and corpus present.")

In [ ]:
import sys

os.chdir(REPO_DIR)
# Build tools first: seqeval is a legacy setup.py package and fails without them.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "numpy", "pandas", "scipy", "openpyxl"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "seqeval", "--no-build-isolation"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
)

os.environ["THESIS_RUN_ENV"] = "colab"
os.environ["WANDB_DISABLED"] = "true"
print("Dependencies OK.")

## 4 — Preflight: is the completed training run actually there?

This notebook only post-processes existing predictions. If the OOF workbooks are missing it stops here, instead of walking the whole Drive mount for twenty minutes and then writing an empty file.

In [ ]:
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from build_unified_protocol_table import discover_oof_files

if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(
        f"{OUTPUT_DIR} does not exist. Set OUTPUT_DIR in cell 2 to the directory the "
        "training run wrote to (the one containing experiment_outputs/)."
    )

candidates = discover_oof_files([OUTPUT_DIR], [], "*oof*.xlsx")
print(f"OOF workbooks under {OUTPUT_DIR}: {len(candidates)}")
for path in candidates[:5]:
    print("   ", path.name)
if len(candidates) > 5:
    print(f"    ... and {len(candidates) - 5} more")

if not candidates:
    raise FileNotFoundError(
        f"No *oof*.xlsx workbooks under {OUTPUT_DIR}. This notebook cannot produce results "
        "on its own — it needs a completed run. The workbooks live under "
        "experiment_outputs/exp06_*_oof/."
    )

if RUNNER_XLSX.is_file():
    print(f"\n{RUNNER_XLSX.name}: found (lambda grid, loss config and holdout sheets will be copied in)")
else:
    print(
        f"\n{RUNNER_XLSX.name}: MISSING — results and p-values will still be built, "
        "but the journal_* reference sheets will be absent."
    )

## 5 — Regenerate the Exp07 split JSONs

The training run wrote these to the ephemeral `/content` disk, so they do not survive a session. Regenerating is exact rather than approximate: `regenerate_exp07_splits()` is deterministic and the runner called it with no overrides, so the same defaults (`split_seed=42`, `split_ratio=0.7`) and the same corpus pickle reproduce byte-identical JSONs.

That exactness is what makes section 6 valid — `sentence_id` is a position within `(train + eval)`, so the ordering has to match what the runner saw. Idempotent: skipped if the files are already on Drive. Seconds, CPU, no training.

In [ ]:
os.chdir(REPO_DIR)

if (SPLITS_DIR / "split_meta.json").is_file():
    print(f"Splits already present at {SPLITS_DIR} — nothing to do.")
else:
    # The two env vars the runner would have set; get_splits_dir() reads the second.
    os.environ["THESIS_NER_CSV"] = str(NER_DATASET_PKL)
    os.environ["THESIS_EXP07_SPLITS_DIR"] = str(SPLITS_DIR)
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)

    for sub in ("experiments", "core"):
        p = str(REPO_DIR / sub)
        if p not in sys.path:
            sys.path.insert(0, p)

    from exp07_split_artifacts import regenerate_exp07_splits

    info = regenerate_exp07_splits(clear_existing=True)
    print("dataset:          ", info.get("dataset"))
    print("source_sentences: ", info.get("source_sentences"))
    print("variants:         ", info.get("variants"))

print("\nFiles in", SPLITS_DIR)
for p in sorted(SPLITS_DIR.glob("*.json")):
    print("   ", p.name)

## 6 — Build `unified_protocol_table.xlsx`

**Why this exists.** Mixing 70/30 holdout numbers with OOF numbers in one table is not just a presentation problem: the OOF outer folds train the base models on ~80% of the corpus while the holdout trains on ~70%, so a mixed table credits fusion with a training-data advantage that has nothing to do with fusion. Scoring everything on the identical pooled outer-test tokens removes that confound.

**`--splits-dir`, not the single-JSON flags.** Fold membership is condition-specific, so the builder resolves each run's own split JSONs by variant rather than assuming one pair for all conditions.

**The slow part is Drive, not compute.** Roughly 900 workbooks have to be read over a FUSE mount. Output streams line by line and is mirrored to `cell9b_unified.log`, so a dropped connection does not lose it.

In [ ]:
os.chdir(REPO_DIR)

argv = [
    sys.executable,
    "build_unified_protocol_table.py",
    "--oof-dir", str(OUTPUT_DIR),
    "--splits-dir", str(SPLITS_DIR),
    "--output", str(UNIFIED_XLSX),
]
if RUNNER_XLSX.is_file():
    argv += ["--reference-xlsx", str(RUNNER_XLSX)]

print("Command:")
for i, part in enumerate(argv):
    print(f"  [{i}] {part}")
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Popen + line iteration, not subprocess.run: run() inherits the raw fd and Colab then
# shows nothing until the process exits, which makes a long Drive scan look like a hang.
log_path = OUTPUT_DIR / "cell9b_unified.log"
with open(log_path, "a", encoding="utf-8") as logf, subprocess.Popen(
    argv,
    cwd=str(REPO_DIR),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
) as proc:
    for line in proc.stdout:
        print(line, end="")
        logf.write(line)
        logf.flush()
    proc.wait()

if proc.returncode != 0:
    raise RuntimeError(
        f"build_unified_protocol_table.py exited with code {proc.returncode}. See {log_path}."
    )

print("\nWrote:", UNIFIED_XLSX)

## 7 — The numbers for the paper

Sheet map: main-table cells → `fold_summary`; headline gain → `fusion_vs_base`; p-values → `paired_fold_tests`; error analysis → `token_error_summary`; Methods λ → `journal_lambda_grid`; holdout comparison → `journal_main_table`.

`split_condition` carries the runner's `__seed<N>` suffix, so there is one per (condition, seed). `base_split_condition` drops it and is the real data condition — that is what the summary and significance sheets group on, so the 20 training seeds pool into one *n*=100 test per contrast instead of 20 tests of *n*=5.

In [ ]:
import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 400)

sheets = pd.read_excel(UNIFIED_XLSX, sheet_name=None)
print("Sheets:")
for name, frame in sheets.items():
    print(f"  {name:28s} {frame.shape[0]:6d} rows x {frame.shape[1]:2d} cols")

summary = sheets["fold_summary"]
print("\n=== main table: mean strict entity F1 over all (seed, fold) units ===")
print(
    summary.pivot_table(
        index=["model", "base_split_condition"], columns="method", values="mean_f1"
    )
    .round(3)
    .to_string()
)
print("\nseeds per cell:", sorted(summary["n_seeds"].unique()))
print("paired units per cell:", sorted(summary["n_units"].unique()))

In [ ]:
tests = sheets["paired_fold_tests"]

print("=== paired_fold_tests: Holm-adjusted within each (model, base_split_condition) family ===")
print(
    tests[[
        "model", "base_split_condition", "contrast", "n_pairs", "mean_delta_f1",
        "pairs_favouring_fusion", "wilcoxon_p_holm", "ttest_rel_p_holm", "sig_01_holm",
    ]].to_string(index=False)
)

# 20 seeds x 5 folds should read 100. If it reads 5 the seeds did not pool, and no
# p-value can fall below 0.0625 regardless of effect size.
print("\nn_pairs observed:", sorted(tests["n_pairs"].unique().tolist()))
print(
    "significant at Holm-corrected alpha=0.01:",
    f"{int(tests['sig_01_holm'].sum())} / {len(tests)}",
)

# A saturated signed-rank test reports its floor, not a measured quantity. Where raw p
# equals min_attainable_wilcoxon_p, report "all N pairs favour fusion" plus the effect size.
at_floor = tests["wilcoxon_p"].round(20) <= tests["min_attainable_wilcoxon_p"].round(20)
print(f"contrasts sitting on the signed-rank floor: {int(at_floor.sum())} / {len(tests)}")

not_sig = tests[~tests["sig_01_holm"]]
if not not_sig.empty:
    print("\nContrasts that do NOT clear alpha=0.01:")
    print(
        not_sig[[
            "model", "base_split_condition", "contrast", "mean_delta_f1",
            "pairs_favouring_fusion", "wilcoxon_p_holm",
        ]].to_string(index=False)
    )

In [ ]:
errors = sheets.get("token_error_summary")
if errors is not None and not errors.empty:
    print("=== token_error_summary: token counts summed over all training seeds ===")
    print("Token-level over BIO tags, while F1 is entity-level with exact span and type.")
    print("The two answer different questions and will not reconcile arithmetically.\n")
    print(
        errors[[
            "model", "base_split_condition", "method", "n_seeds", "eval_tokens",
            "entity_tokens", "false_positive", "false_negative", "type_error",
            "boundary_error", "token_error_rate",
        ]].to_string(index=False)
    )
else:
    print("token_error_summary is empty — classify_error() could not be imported.")

## 8 — Where everything landed

In [ ]:
for label, path in [
    ("unified table (write the paper from this)", UNIFIED_XLSX),
    ("runner export (source of record, keep it)", RUNNER_XLSX),
    ("build log", OUTPUT_DIR / "cell9b_unified.log"),
    ("split JSONs", SPLITS_DIR),
]:
    if path.is_dir():
        print(f"{label}:\n  {path}  ({len(list(path.glob('*')))} files)")
    elif path.is_file():
        print(f"{label}:\n  {path}  ({path.stat().st_size:,} bytes)")
    else:
        print(f"{label}:\n  {path}  (missing)")